# Train a pothole detector → export to TFLite (Android)

This notebook trains a YOLOv8 pothole detector and exports it as a `.tflite` file the mobile app will load.

**Two paths:**
1. **Quick win (Section 3)** — export the generic YOLOv8n in ~30 seconds. Useful for wiring up the phone app before a real model exists.
2. **Real detector (Section 4)** — train YOLOv8n on a pothole dataset, then export.

## Important: Colab is throwaway
Everything in `/content/` is **wiped on disconnect/reset**. Only files copied to `My Drive/` survive. This notebook saves `best.pt` and the `.tflite` to your Drive at key checkpoints so a disconnect won't lose your trained model.

## First: enable a GPU runtime
**Runtime → Change runtime type → T4 GPU → Save.** Then run cells in order with **Shift+Enter** (or click ▶ next to each cell).

In [ ]:
!nvidia-smi

## 1. Install Ultralytics YOLO
Installs the YOLO library plus the Roboflow client (used in Section 4).

In [ ]:
%pip install -q ultralytics roboflow

## 2. Mount Google Drive
Saves trained models here so a Colab disconnect doesn't lose your work.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/road-quality-visualizer'
os.makedirs(SAVE_DIR, exist_ok=True)
print('Files will be saved to:', SAVE_DIR)

## 3. Quick win — export the generic YOLOv8n to TFLite

Proves the export pipeline works in seconds. The result detects general objects (people, cars, …), **not** potholes — useful only for wiring up the phone app before you have a real pothole model.

If you only want a real pothole detector, skip to Section 4.

In [ ]:
from ultralytics import YOLO
import shutil, glob

m = YOLO('yolov8n.pt')
m.export(format='tflite')

candidates = glob.glob('yolov8n_saved_model/*float32.tflite') or glob.glob('yolov8n_saved_model/*.tflite')
for f in candidates[:1]:
    dest = f'{SAVE_DIR}/yolov8n.tflite'
    shutil.copy(f, dest)
    print('Saved', dest)

## 4. Train a real pothole detector

You need a labeled pothole dataset in YOLOv8 format. Easiest source: **Roboflow Universe** (free).

### 4a. Get a pothole dataset from Roboflow
1. Sign up free at https://roboflow.com (you can use your Google account).
2. Go to https://universe.roboflow.com and search **pothole**.
3. Pick a project — look for one with hundreds of images and clean labels.
4. Click **Download Dataset** → choose **YOLOv8** format → **Show download code** → it gives a small Python snippet.
5. Paste the snippet into the next cell (replacing the placeholder).

In [ ]:
# Paste your Roboflow snippet here. It will look roughly like:
#
#   from roboflow import Roboflow
#   rf = Roboflow(api_key='YOUR_API_KEY')
#   project = rf.workspace('workspace-name').project('project-name')
#   version = project.version(1)
#   dataset = version.download('yolov8')
#
# The next cell automatically reads `dataset.location`, so you don't need to hardcode any path.

### Alternative: upload your own YOLO-format dataset zip
Already have a dataset packed as a `.zip` (containing `images/`, `labels/`, and `data.yaml`)? Uncomment this cell to upload it from your computer instead of using Roboflow.

In [ ]:
# from google.colab import files
# import types
# up = files.upload()
# fn = list(up.keys())[0]
# !unzip -q -o "$fn" -d /content/dataset
# dataset = types.SimpleNamespace(location='/content/dataset')
# print('Dataset extracted to:', dataset.location)

### Set the dataset config path
This automatically uses `dataset.location` from whichever source you ran above (Roboflow snippet or upload cell). No hardcoded paths needed.

In [ ]:
DATA_YAML = f'{dataset.location}/data.yaml'
print('Using dataset config:', DATA_YAML)
!ls -la "{dataset.location}"

### 4b. Train
Fine-tunes YOLOv8n on your pothole dataset. A few thousand images on a T4 GPU takes ~1–3 hours for 50 epochs. Lower `epochs` for a faster (but weaker) first try.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')              # start from the pretrained nano model
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    project='runs',
    name='potholes',
)

### Back up `best.pt` to Drive *right after training*
Before running evaluation/inference/export, save the trained weights to Drive — so even if the session dies in the next step, the hard work survives.

In [ ]:
import shutil, glob
best = next(iter(glob.glob('runs/**/best.pt', recursive=True)), None)
if best:
    shutil.copy(best, f'{SAVE_DIR}/pothole_best.pt')
    print('Backed up best.pt to Drive →', f'{SAVE_DIR}/pothole_best.pt')
else:
    print('best.pt not found yet — did training finish? (Looks under runs/**/.)')

### 4c. Evaluate on the validation set
`mAP50` and `mAP50-95` are detection accuracy scores (higher = better).

In [ ]:
metrics = model.val(data=DATA_YAML)
print('mAP50   :', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

### 4d. Sanity inference on a few test images
Confirms the model draws boxes around potholes. Annotated images appear under `runs/detect/predict*/`. Tries `test/`, `valid/`, then `val/` automatically — datasets use different folder names.

In [ ]:
import glob, os
for sub in ('test/images', 'valid/images', 'val/images'):
    d = os.path.join(dataset.location, sub)
    sample = glob.glob(f'{d}/*')[:5]
    if sample:
        print('Predicting on:', d)
        model.predict(source=sample, save=True, conf=0.35)
        print('Boxed images saved under runs/detect/.')
        break
else:
    print('No test/ or valid/ images found — check your dataset layout.')

### 4e. Export the trained model

Exports **two** formats from the in-memory model (no disk reload):
- **TFLite** — for Android on-device inference (`pothole.tflite`)
- **ONNX** — for backend / server-side inference (`pothole.onnx`)

Both get copied to your Drive so you can download whichever you need.

In [ ]:
import shutil, glob

# model.export() returns the actual output path — no need to glob/guess the name.
# - TFLite for Android on-device inference
# - ONNX for backend / server-side inference
tflite_path = model.export(format='tflite')
onnx_path = model.export(format='onnx')

print('Export returned:')
print('  TFLite:', tflite_path)
print('  ONNX  :', onnx_path)

# Find the actual best.pt path (Ultralytics may use runs/detect/potholes/... or runs/potholes/...).
best_candidates = glob.glob('runs/**/best.pt', recursive=True)
best = best_candidates[0] if best_candidates else None
print('best.pt path:', best)

# TFLite: the path can be a folder (saved_model) or a file. Find the float32 .tflite inside.
import os
if os.path.isdir(tflite_path):
    tfs = glob.glob(f'{tflite_path}/*float32.tflite') or glob.glob(f'{tflite_path}/*.tflite')
    tflite_file = tfs[0] if tfs else None
else:
    tflite_file = tflite_path

if tflite_file:
    shutil.copy(tflite_file, f'{SAVE_DIR}/pothole.tflite')
    print('Saved TFLite →', f'{SAVE_DIR}/pothole.tflite')

# ONNX is a single file.
shutil.copy(onnx_path, f'{SAVE_DIR}/pothole.onnx')
print('Saved ONNX  →', f'{SAVE_DIR}/pothole.onnx')

# Keep the original PyTorch weights as a backup.
if best:
    shutil.copy(best, f'{SAVE_DIR}/pothole_best.pt')
    print('Saved PyTorch weights →', f'{SAVE_DIR}/pothole_best.pt')

## Done
Your files are in Google Drive at `My Drive/road-quality-visualizer/`:

- `pothole.tflite` — Android on-device inference (drop in `mobile/android/app/src/main/assets/`)
- `pothole.onnx` — backend server-side inference (drop in `backend/models/`)
- `pothole_best.pt` — PyTorch weights (backup; for re-export or fine-tuning later)

Download whichever you need.

---

## If a session disconnects during training
Reconnect, then **Runtime → Run all** from the top. Roboflow will re-download in a few seconds, but training itself starts over. To minimize the risk:
- Keep this tab open (minimized is fine).
- Stop the laptop from sleeping (Windows: Settings → Power → "Never").
- Glance at the tab occasionally — any interaction resets the ~90-min idle timer.

---

## Re-export shortcut (skip training)

If `pothole_best.pt` is **already in your Drive** and you just want fresh
`.tflite` / `.onnx` files (e.g. after the notebook was updated, or you want a
different format), **skip the training cells entirely** and run the cell below.

Prerequisites:
- Run **Section 1** (`%pip install ultralytics roboflow`).
- Run **Section 2** (mount Drive, set `SAVE_DIR`).

Then just hit ▶ on the next cell. Takes ~30 seconds (no GPU needed).

In [ ]:
from ultralytics import YOLO
import shutil, os, glob

# Load saved weights (no training) and re-export to TFLite + ONNX.
model = YOLO(f'{SAVE_DIR}/pothole_best.pt')
tflite_path = model.export(format='tflite')
onnx_path = model.export(format='onnx')

# TFLite path may be a folder (saved_model) or a single file.
if os.path.isdir(tflite_path):
    tfs = glob.glob(f'{tflite_path}/*float32.tflite') or glob.glob(f'{tflite_path}/*.tflite')
    tflite_file = tfs[0] if tfs else None
else:
    tflite_file = tflite_path

if tflite_file:
    shutil.copy(tflite_file, f'{SAVE_DIR}/pothole.tflite')
    print('Saved TFLite →', f'{SAVE_DIR}/pothole.tflite')

shutil.copy(onnx_path, f'{SAVE_DIR}/pothole.onnx')
print('Saved ONNX  →', f'{SAVE_DIR}/pothole.onnx')
print('Done.')